# Introduction to Polymorphic Functions

- **Monomorphic functions** operate on a single, specific data type.
- When writing **higher-order functions (HOFs)**, we aim to design code that works **independently of the data type** it is given.
- This raises an important question: **How can we achieve this level of generality?**
- **Polymorphic functions** emerge by identifying common structural patterns shared across multiple monomorphic functions.



## Example: `findFirst` Function

- Searches through a list of `String` values
- Looks for a specific `String` key
- Returns:
  - The index of the **first occurrence** of the key
  - `-1` if the key is **not found**

In [ ]:
fun findFirst(ss: Array<String>, key: String): Int {
    for (i in ss.indices) {
        if (ss[i] == key) {
            return i
        }
    }
    return -1
}

In [ ]:
fun main() {
    val fruits = arrayOf("apple", "cherry", "date", "apple")

    val indexCherry = findFirst(fruits, "cherry")
    val indexGrapes = findFirst(fruits, "grapes")

    println("Index of 'cherry': $indexCherry")
    println("Index of 'grapes': $indexGrapes")
}

main()


Index of 'cherry': 1
Index of 'grapes': -1


> **Remember:** Traditional loops can be expressed *functionally* using **tail-call recursion**, where the loop state is passed explicitly through function parameters.


In [ ]:
fun findFirst(ss: Array<String>, key: String): Int {
    tailrec fun loop(n: Int): Int =
        when {
            n >= ss.size -> -1
            ss[n] == key -> n
            else -> loop(n + 1)
        }
    return loop(0)
}

In [ ]:
fun main() {
    val fruits = arrayOf("apple", "cherry", "date", "apple")

    val indexCherry = findFirst(fruits, "cherry")
    val indexGrapes = findFirst(fruits, "grapes")

    println("Index of 'cherry': $indexCherry")
    println("Index of 'grapes': $indexGrapes")
}

main()

Index of 'cherry': 1
Index of 'grapes': -1


### Example: Searching in a List of Integers

Suppose you want to search for the index of the values **3** and **1000** in the list: 1, 3, 2, 3, 40.

The code looks as follows:


In [ ]:
fun findFirst(nums: Array<Int>, key: Int): Int {
    tailrec fun loop(n: Int): Int =
        when {
            n >= nums.size -> -1
            nums[n] == key -> n
            else -> loop(n + 1)
        }
    return loop(0)
}

In [ ]:
fun main() {
    val numbers = arrayOf(1, 3, 2, 3, 40)

    val indexThree = findFirst(numbers, 3)
    val indexThousand = findFirst(numbers, 1000)

    println("Index of 3: $indexThree")
    println("Index of 1000: $indexThousand")
}
main()

Index of 3: 1
Index of 1000: -1


### Key Observations

- This version works **only with `Int` values**
- Its **structure is identical** to the `String` version
- Both implementations are **monomorphic functions**
- Recognizing this **shared structure** motivates the need for **polymorphic functions**

## Example of a polymorphic function

- The specific details of the code are **not the main focus** here.
- What matters is that the structure of `findFirst` is **almost identical** whether we are:
  - Searching for a `String` in an `Array<String>`
  - Searching for an `Int` in an `Array<Int>`
  - Searching for a value of type `A` in an `Array<A>`
- This observation highlights a **shared computational pattern** across different data types.
- **Figure 2.2** demonstrates how `findFirst` can be written **more generally** for any type `A`
- This general version works by **accepting a function** that tests whether a given value of type `A` matches the search criteria.


![image.png](attachment:0424c690-ae75-4999-af82-c4c3d8366d31.png)

In [ ]:
fun <A> findFirst(xs: Array<A>, p: (A) -> Boolean): Int {
    tailrec fun loop(n: Int): Int =
        when {
            n >= xs.size -> -1
            p(xs[n]) -> n
            else -> loop(n + 1)
        }
    return loop(0)
}


- This is an example of a **polymorphic (generic) function**
- The function **abstracts over**:
  - The type of elements stored in the array
  - The function used to test each element
- Polymorphic functions introduce **type parameters** using angle brackets (e.g., `<A>`) after the function name
- Type parameter names are flexible, but by convention we use **short, uppercase letters** such as `<A, B, C>`

- Type parameters introduce **type variables** that can be used throughout the function’s type signature
- In `findFirst`, the type variable `A` is used:
  - As the element type of the array (`Array<A>`)
  - As the input type to the predicate function (`(A) -> Boolean`)
- Because the same type variable appears in both places, the compiler **enforces type consistency**
- As a result, attempting to search for a `String` in an `Array<Int>` will produce a **type mismatch error**


In [ ]:
fun main(){
    val fruits = arrayOf("apple", "cherry", "date", "apple")

    val indexCherry = findFirst(fruits) { it == "cherry" }
    val indexGrapes = findFirst(fruits) { it == "grapes" }

    println("Index of 'cherry': $indexCherry")
    println("Index of 'grapes': $indexGrapes")
}
main()

Index of 'cherry': 1
Index of 'grapes': -1


In [ ]:
val numbers = arrayOf(1, 3, 2, 3, 40)

val indexThree = findFirst(numbers) { it:Int -> it == 3 }
val indexThousand = findFirst(numbers) { it == 1000 }

println("Index of 3: $indexThree")
println("Index of 1000: $indexThousand")

Index of 3: 1
Index of 1000: -1


### Exercise

In [ ]:
val xs = listOf(1, 2, 3, 4)
xs

[1, 2, 3, 4]

In [ ]:
// get frist element
xs.first() //1

1

In [ ]:
//drop - remove the first n element
xs.drop(1)

[2, 3, 4]

In [ ]:
xs.drop(2)

[3, 4]

###  Extension methods and properties

Kotlin provides us with a convenient way of adding behavior (or state) to any type of instance. It does so by way of extension methods and properties.

We can easily add behavior to all instances of a given type by adding an extension method as follows:

In [ ]:
fun Int.show(): String = "The value of this Int is $this"


In [ ]:
1.show()

The value of this Int is 1

Extending `List` with `head` and `tail`

Just as we added new behavior to Int using **extension methods and properties**, we can do the same for lists.

In functional programming, it is common to think of a list as being composed of:

- a **head** (the first element), and

- a **tail** (everything after the first element)

Using Kotlin extension properties, we can add this behavior to any `List<T>`.

In [ ]:
val <T> List<T>.tail: List<T>
    get() = drop(1)

val <T> List<T>.head: T
    get() = first()

In [ ]:
val xs = listOf(1, 2, 3, 4, 5)


In [ ]:
xs.head

1

In [ ]:
xs.tail

[2, 3, 4, 5]

In [ ]:
val <T> List<T>.tail: List<T>
    get() = drop(1)

val <T> List<T>.head: T
    get() = first()

fun <A> isSorted(aa: List<A>, order: (A, A) -> Boolean): Boolean {
    tailrec fun go(x: A, xs: List<A>): Boolean =
        if (xs.isEmpty()) true
        else if (!order(x, xs.head)) false
        else go(xs.head, xs.tail)

    return aa.isEmpty() || go(aa.head, aa.tail)
}

In [ ]:
fun main() {
    val xs = listOf(1, 2, 3, 4, 5)
    val ys = listOf(5, 4, 3, 2, 1)
    val words = listOf("apple", "banana", "cherry", "date")

    // Ascending: each element should be <= the next
    val ascending = isSorted(xs) { a, b -> a <= b }
    val ascending2 = isSorted(ys) { a, b -> a <= b }

    // Ascending (alphabetical order)
    val asc = isSorted(words) { a, b -> a <= b }

    // Descending: each element should be >= the next
    val descending = isSorted(ys) { a, b -> a >= b }
    val descending2 = isSorted(xs) { a, b -> a >= b }

    println("xs ascending? $ascending")     // true
    println("ys ascending? $ascending2")    // false
    println("words Ascending? $asc")        // true
    println("ys descending? $descending")   // true
    println("xs descending? $descending2")  // false
}
main()

xs ascending? true
ys ascending? false
words Ascending? true
ys descending? true
xs descending? false
